## Задачи для PyTorch

In [42]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchtext
import random
import matplotlib.pyplot as plt
from gensim.models import Word2Vec
import torchtext.data as data
from torchtext.data import Field, LabelField
from torchtext.data import BucketIterator
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, roc_curve, auc

# Проверка доступности GPU
device ="cpu"# torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f'Using device: {device}')

# Загрузка датасета
df = pd.read_csv('FakeNewsNet.csv')  # Замените на путь к вашему файлу
df = df[['title', 'news_url', 'source_domain', 'real']]

# Проверка на null-значения
print(df.isnull().sum())
df.dropna(inplace=True)

# Проверка баланса классов
print(df['real'].value_counts())

# Комбинирование текстов
df['text'] = df['title'] + ' ' + df['news_url'] + ' ' + df['source_domain']
df = df[['text', 'real']]

# Определение полей TEXT и LABEL
TEXT = 'text'
LABEL = 'real'

# Создание датасета
dataset = df[[TEXT, LABEL]]

# Определите поля (fields)
text_field = data.Field(sequential=True, tokenize='spacy', lower=True, tokenizer_language='en_core_web_sm')
label_field  = data.LabelField(dtype=torch.float)

# Построение словарей
fields = [(TEXT, text_field), (LABEL, label_field)]
dataset = data.TabularDataset(fields=fields)

# Разделение данных на тестовую и обучающую выборки
train_data, test_data = train_test_split(dataset, test_size=0.2, random_state=42)

# Создание итераторов
train_iterator, test_iterator = data.BucketIterator.splits(
    (train_data, test_data),
    batch_size=64,
    sort_key=lambda x: len(x.text),
    sort_within_batch=True,
    device=device
)

# Определение BATCH_SIZE
BATCH_SIZE = 30

Using device: cpu
title              0
news_url         330
source_domain    330
real               0
dtype: int64
real
1    17371
0     5495
Name: count, dtype: int64


TypeError: TabularDataset.__init__() missing 2 required positional arguments: 'path' and 'format'

In [39]:
# Соединяем обучающие и тестовые данные
all_texts = train_data.text + test_data.text

# Обучаем Word2Vec на всех данных
w2v_model = Word2Vec(sentences=all_texts, vector_size=EMBEDDING_DIM, window=5, min_count=5, workers=4)
w2v_model.build_vocab(all_texts)
w2v_model.train(all_texts, total_examples=len(all_texts), epochs=10)
w2v_model.save('w2v_model.txt')

# Загружаем обученную модель
w2v_model = Word2Vec.load('w2v_model.txt')
pre_trained_emb = torch.FloatTensor(w2v_model.wv.vectors)

# Создаём итератор
train_iterator, test_iterator = BucketIterator.splits(
    (train_data, test_data),
    batch_size=BATCH_SIZE,
    device=device,
    sort_within_batch=True,
    sort_key=lambda x: len(x.text))

TypeError: unsupported operand type(s) for +: 'generator' and 'generator'

In [30]:
class RNN(nn.Module):
    def __init__(self, pre_trained_emb, embedding_dim, hidden_dim, output_dim, n_layers, dropout):
        super(RNN, self).__init__()

        # Эмбеддинг
        self.embedding = nn.Embedding.from_pretrained(pre_trained_emb)

        # RNN-слой
        self.rnn = nn.RNN(embedding_dim, hidden_dim, num_layers=n_layers, dropout=dropout, batch_first=True)

        # Линейный слой для предсказания
        self.fc = nn.Linear(hidden_dim, output_dim)

        # Функция активации
        self.dropout = nn.Dropout(dropout)

    def forward(self, text):
        # Эмбеддинг входных данных
        embedded = self.embedding(text)

        # Проход через RNN-слой
        output, hidden = self.rnn(embedded)

        # Извлечение последнего скрытого состояния
        hidden = hidden[-1, :, :]

        # Применение дропаута
        output = self.dropout(hidden)

        # Предсказание
        output = self.fc(output)

        return output

In [31]:
# Определяем параметры модели
EMBEDDING_DIM = 300  # Размерность эмбеддинга (из word2vec)
HIDDEN_DIM = 128    # Размерность скрытого состояния RNN
OUTPUT_DIM = 1      # Размерность выходного слоя (1 для бинарной классификации)
N_LAYERS = 2        # Количество слоев в RNN
DROPOUT = 0.5      # Вероятность дропаута

In [32]:
# Создание модели
model = RNN(pre_trained_emb, EMBEDDING_DIM, HIDDEN_DIM, OUTPUT_DIM, N_LAYERS, DROPOUT)

optimizer = optim.Adam(model.parameters())
criterion = nn.BCEWithLogitsLoss()

# Обучение модели
def train(model, iterator, optimizer, criterion):
    epoch_loss = 0
    model.train()
    for batch in iterator:
        # Замена неизвестных слов на UNK
        batch.text = batch.text.masked_fill(batch.text >= len(TEXT.vocab), 0)
        optimizer.zero_grad()
        text = batch.text
        label = batch.label
        predictions = model(text)
        loss = criterion(predictions, label)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    return epoch_loss / len(iterator)

# Тестирование модели
def evaluate(model, iterator, criterion):
    epoch_loss = 0
    model.eval()
    with torch.no_grad():
        for batch in iterator:
            text = batch.text
            label = batch.label
            predictions = model(text)
            loss = criterion(predictions, label)
            epoch_loss += loss.item()
    return epoch_loss / len(iterator)
    

In [33]:
# Тренировка
N_EPOCHS = 10
for epoch in range(N_EPOCHS):
    train_loss = train(model, train_iterator, optimizer, criterion)
    test_loss = evaluate(model, test_iterator, criterion)
    print(f'Epoch: {epoch+1:02} | Train Loss: {train_loss:.3f} | Test Loss: {test_loss:.3f}')



IndexError: index out of range in self

In [37]:
print(f"Размер словаря: {len(TEXT.vocab)}")
print(f"Размер эмбеддинга: {pre_trained_emb.shape[0]}")

print(f"Вектор UNK: {pre_trained_emb[0]}") 

Размер словаря: 35945
Размер эмбеддинга: 5413
Вектор UNK: tensor([-0.5826, -0.8312,  0.3694,  0.0421, -1.0749, -0.0608,  0.9786,  1.2207,
        -0.8561, -0.5949,  0.0425, -0.8407, -0.6343, -0.3425, -0.4814, -0.4318,
        -0.4466,  0.1267,  0.0597, -0.4209,  0.2630,  0.6615, -0.3787,  0.1635,
        -0.1804, -0.8651, -0.2411, -0.0639, -0.0570,  0.2933,  0.3953, -0.8699,
         0.2840,  0.1260,  0.1524,  0.1880, -0.6110, -1.1692,  0.5900,  0.8115,
         0.2629, -0.4453,  0.0821, -0.4119, -0.0038,  0.0460, -0.7205, -0.2192,
         0.4687,  0.9568,  0.3406,  0.1812,  0.8923, -0.0019, -0.3999,  0.1181,
        -0.9701,  0.0872,  0.7797,  0.1442, -0.3965, -0.0840,  0.3569,  0.2633,
         1.1461, -0.7595,  0.1154,  0.2420,  0.3593, -0.2513,  0.1836,  0.2940,
         0.1232, -0.0737, -0.0310, -0.9497, -0.4915, -0.5428,  0.1156,  0.6997,
        -1.1371, -0.6772,  0.1450,  0.3815, -0.1995, -0.6673, -0.8948,  0.2286,
        -0.8019,  0.3444, -0.2821,  0.3091,  0.4799,  0.2172, 

In [23]:
print(batch.text)

NameError: name 'batch' is not defined